In [ ]:
from formula_engine.file_reader import read_parquet
from formula_engine.processes.dataframe import TopNLongestTripsDataFrameProcess
from formula_engine.processes.sql import TopNLongestTripsSqlProcess
from formula_engine.types import Process, DataFrame

from memory_profiler import profile
from perf_timer import PerfTimer


@PerfTimer("main")
@profile
def main(input_df: DataFrame, process: Process) -> DataFrame:
    df = process.execute(input_df)
    print(df.schema())
    df.show()
    return df


if __name__ == "__main__":
    input_df = read_parquet(
        "/home/mac/job/bigdata/data/nytaxi/yellow_tripdata_*.parquet"
    )

    process_df = TopNLongestTripsDataFrameProcess(10000)
    process_sql = TopNLongestTripsSqlProcess(6)

    result_df = main(input_df, process_df)

    result_df.write_parquet("/home/mac/job/bigdata/data/output/longest_trips_data")

In [ ]:
from daft import col, DataFrame


def top3_longest_trips_in_time(df: DataFrame, n: int) -> DataFrame:
    return (
        df.select("tpep_pickup_datetime", "tpep_dropoff_datetime")
        .with_column(
            "trip_duration", col("tpep_dropoff_datetime") - col("tpep_pickup_datetime")
        )
        .sort(col("trip_duration"), desc=True)
        .limit(n)
    )


class TopNLongestTripsDataFrameProcess:
    def __init__(self, n: int):
        self.n = n

    def execute(self, df: DataFrame) -> DataFrame:
        return top3_longest_trips_in_time(df, self.n)


In [ ]:
from daft import DataFrame
from typing import Protocol


class Process(Protocol):
    def execute(self, df: DataFrame) -> DataFrame: ...


In [ ]:
import daft


def read_parquet(file_path: str) -> daft.DataFrame:
    return daft.read_parquet(file_path)


: 

In [ ]:
from daft import DataFrame, sql


def top3_longest_trips_in_time(df: DataFrame, n: int) -> DataFrame:
    return sql(f"""
        SELECT tpep_pickup_datetime, tpep_dropoff_datetime, (tpep_dropoff_datetime - tpep_pickup_datetime) AS trip_duration
        FROM df
        ORDER BY trip_duration DESC
        LIMIT {n}
    """)


class TopNLongestTripsSqlProcess:
    def __init__(self, n: int):
        self.n = n

    def execute(self, df: DataFrame) -> DataFrame:
        return top3_longest_trips_in_time(df, self.n)
